<a href="https://colab.research.google.com/github/oooinr4018-web/-1/blob/main/ESAA_0904_%EC%88%98%EC%83%81%EC%9E%91_%EB%A6%AC%EB%B7%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 주제

- 생활 습관 및 건강 지표를 활용한 건강 상태 분류

수면 시간, 심박수, BMI, 칼로리 소모량, 걸음 수, 운동 시간, 수분 섭취량 등의 수치형 변수와 식단, 스트레스 수준, 수면의 질, 신체 활동 수준, 흡연/음주 여부, 성별 등의 범주형 변수를 이용해 사용자의 건강 상태를 at-risk, unhealthy, fit 중 하나로 예측

# 데이터

train.csv

1. 데이터 수: 690,088개

2. Target: health_condition

- at-risk

- unhealthy

- fit

*클래스별 데이터 수가 at-risk 692,561개, unhealthy 57,724개, fit 39,803개 -> 클래스 불균형이 큰 데이터

3. 수치형 변수

- sleep_duration

- heart_rate

-bmi

- calorie_expenditure

- step_count

- exercise_duration

- water_intake

4. 범주형 변수

- diet_type

- stress_level

- sleep_quality

- physical_activity_level

- smoking_alcohol

- gender

# 코드 흐름

1. 데이터 확인 및 Train / Validation 분리

- 데이터의 자료형, 결측치 확인

- health_condition을 target으로 설정

- train_test_split()의 stratify=y를 사용하여 각 클래스의 비율을 유지한 채 8:2로 데이터 분할

- 예측에 필요하지 않은 id 컬럼 제거

2. 데이터 전처리 (결측치 처리)

- 수치형 변수 -> 평균값으로 대체

- 범주형 변수 - "others"로 대체



In [ ]:
num_imputer=SimpleImputer(strategy="mean")
obj_imputer=SimpleImputer(strategy="constant", fill_value="others")

X_train[num_cols]=num_imputer.fit_transform(X_train[num_cols])
X_train[obj_cols]=obj_imputer.fit_transform(X_train[obj_cols])

- 수치형 변수

: SandardScaler를 이용하여 표준화

- 범주형 변수

: OneHotEncoder를 사용하여 숯 형태로 변환

: drop='first'를 통해 기준 범주 하나를 제거

3. Target Encoding

문자열 형태였던 건강 상태를 모델이 학습할 수 있도록 숫자로 변환



In [ ]:
mapping={
    "at-risk":0,
    "unhealthy": 1,
    "fit": 2
}

y_train_encoded=y_train.map(mapping)

4. 모델 비교

두 가지 분류 모델 학습

- Logistic Regression

- Random Forest Classifier

In [ ]:
models=[
    LogisticRegression(max_iter=5000, random_state=42),
    RandomForestClassifier(random_state=42)
]

- 각 모델을 학습한 후 정확도를 비교하여 성능이 높은 모델을 best_model로 선정

- Random Forest가 학습 데이터에서 가장 높은 정확도를 보여 최종 모델로 선택.

- 학습 데이터 정확도가 거의 1에 가까웠기 때문에 과적합 가능성을 확인할 필요가 있음.



5. Test 데이터 성능 평가

Train 데이터에서 학습한

- Imputer

- StandardScaler

- OneHotEncoder

를 Test 데이터에 동일하게 적용하고 최종 모델로 예측.

Test 데이터 성능은

- Accuracy=0.97

- Macro F1-score=0.91

- Weighted F1-score=0.96

6. 최종 예측 파일 생성

Competition test 데이터에도 동일한 전처리를 적용한 후 Random Forest로 건강 상태를 예측. 숫자로 예측된 label을 다시 at-risk, unhealthy, fit으로 변환하여 id와 결합.

In [ ]:
results_df=pd.oncat([X_test1['id'], y_pred_test1], axis=1)
results_df.to_csv("preddict_healty_risk.csv", index=False)

# 새롭게 알게 된 내용

- Train 데이터에서 학습한 전처리기를 Test 데이터에는 fit_transform이 아니라 transform()만 사용해야 한다는 점

Train/Test 각각에 fit()을 하면 서로 다른 기준으로 전처리되기 때문에, Train 데이터에서 학습한 Imputer, Scaler, Encoder를 Test 데이터에 그대로 적용해야 함.

- 수치형 변수와 범주형 변수는 각각 결측치 처리, 인코딩 방법을 다르게 적용해야 한다는 점

- 다중 분류에서는 Accuracy뿐만 아니라 Precision, Recall, F1-score, Macro Average를 함께 확인해야 한다는 점

# 어려웠던 내용

- 결측치가 많은 데이터에서 수치형, 범주형 변수를 구분하여 서로 다른 방식으로 전처리하는 과정이 복잡했음.

- One-Hot-Encoding 이후 Train/TEst의 컬럼 구조, 순서를 동일하게 유지하는 과젇ㅇ이 중요하다는 점이 어려웠음.

- at-risk 클래스가 대부분을 차지하는 불균형 데이터이기 때문에 단순 Accuracy만으로 모델 성능을 판단하기 어렵기에, 여러 값을 비교해야 한다는 점이 조금 복잡했음.



# 배울 점

- Random Forest는 높은 성능을 보였지만, 학습 데이터 정확도가 거의 100%였기 때문에 학습 성능만 보고 모델을 선택하면 과적합을 놓칠 수 있음.

- 이 코드에서는 Train accuracy를 기준으로 최적 모델을 선택했지만, 실제 모델에서는 Validation 성능, Cross Validation의 Macro F1-score 등을 기준으로 비교하는 방법이 더 적절할 수 있음.

- 클래스 불균형이 큰 문제에서는 Accuracy뿐만 아니라 클래스별 Recall/F1-score, Macro F1-score를 확인하는 것이 중요함.

